# TE02_004 - Live Command Accuracy Validation

This notebook validates TE02_004 by subscribing to ROS 2 topics while a rosbag is played from a terminal.

KPI requirement from `details.txt`: precise execution of commands with 100% accuracy compared to the official remote controller.

## Workflow

1. Start the ROS 2 environment in a terminal.
2. Play the rosbag in the terminal.
3. Run this notebook while the bag is playing.
4. The notebook subscribes to the command and feedback topics and writes the KPI result CSV files.

Suggested terminal command:

```bash
ros2 bag play {$HOME}/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_001/result/WBag/bag_20260707_121849
```

If you need deterministic replay timing, use `--rate 1.0`. If your system uses simulated time, also play with `--clock` and set `USE_SIM_TIME = True` below.

## Validation Method

The notebook compares the official remote-controller command topic with the firmware/executed command topic. For each official command sample, the nearest executed command sample is selected within a configurable time window.

The KPI passes only when all matched command vectors are equal element-by-element and all command vectors have the expected shape. If one of the command topics is missing, the result is `INCONCLUSIVE` rather than `FAIL`, because the bag does not contain enough evidence for the 100% accuracy claim.

In [ ]:
from pathlib import Path
import time
import os

import numpy as np
import pandas as pd
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray, Int8MultiArray, UInt8

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004')
KPI_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = KPI_DIR / 'te02_004_live_command_accuracy_summary.csv'
MATCHED_COMMAND_CSV = KPI_DIR / 'te02_004_live_matched_commands.csv'
RAW_COMMAND_CSV = KPI_DIR / 'te02_004_live_raw_commands.csv'
SUPPORTING_SIGNALS_CSV = KPI_DIR / 'te02_004_live_supporting_signals.csv'

# Default mapping. Swap these two if your bag uses the opposite direction.
OFFICIAL_REMOTE_TOPIC = '/telejoy'
EXECUTED_COMMAND_TOPIC = '/telejoyout'

STATUS_TOPIC = '/statusjoy'
JOINTS_TOPIC = '/joints'
ENCODERS_TOPIC = '/encoders'
JOINT_STATES_TOPIC = '/joint_states'

COMMAND_VECTOR_LENGTH = 6
COMMAND_MATCH_WINDOW_MS = 100.0
ACQUISITION_DURATION_S = 170.0
USE_SIM_TIME = False

print(f'Output directory: {KPI_DIR}')
print(f'Official remote topic: {OFFICIAL_REMOTE_TOPIC}')
print(f'Executed command topic: {EXECUTED_COMMAND_TOPIC}')

Output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004
Official remote topic: /telejoy
Executed command topic: /telejoyout


In [2]:
class LiveCommandAccuracyLogger(Node):
    def __init__(self):
        super().__init__('te02_004_live_command_accuracy_logger')
        if USE_SIM_TIME:
            self.set_parameters([rclpy.parameter.Parameter('use_sim_time', rclpy.Parameter.Type.BOOL, True)])

        self.command_rows = []
        self.status_rows = []
        self.joints_rows = []
        self.encoders_rows = []
        self.joint_state_rows = []

        self.create_subscription(Int8MultiArray, OFFICIAL_REMOTE_TOPIC, self.official_cb, 10)
        self.create_subscription(Int8MultiArray, EXECUTED_COMMAND_TOPIC, self.executed_cb, 10)
        self.create_subscription(UInt8, STATUS_TOPIC, self.status_cb, 10)
        self.create_subscription(Float32MultiArray, JOINTS_TOPIC, self.joints_cb, 10)
        self.create_subscription(Float32MultiArray, ENCODERS_TOPIC, self.encoders_cb, 10)
        self.create_subscription(JointState, JOINT_STATES_TOPIC, self.joint_state_cb, 10)
        self.create_timer(5.0, self.health_cb)

    def now_ns(self):
        return self.get_clock().now().nanoseconds

    def append_command(self, topic, role, data):
        now_ns = self.now_ns()
        values = [int(v) for v in data]
        self.command_rows.append({
            'time_ns': now_ns,
            'time_s': now_ns / 1e9,
            'topic': topic,
            'role': role,
            'command': values,
            'command_len': len(values),
            'valid_shape': len(values) == COMMAND_VECTOR_LENGTH and all(-128 <= v <= 127 for v in values),
        })

    def official_cb(self, msg):
        self.append_command(OFFICIAL_REMOTE_TOPIC, 'official_remote', msg.data)

    def executed_cb(self, msg):
        self.append_command(EXECUTED_COMMAND_TOPIC, 'executed_command', msg.data)

    def status_cb(self, msg):
        now_ns = self.now_ns()
        self.status_rows.append({'time_ns': now_ns, 'time_s': now_ns / 1e9, 'value': int(msg.data)})

    def joints_cb(self, msg):
        now_ns = self.now_ns()
        self.joints_rows.append({'time_ns': now_ns, 'time_s': now_ns / 1e9, 'values': list(msg.data)})

    def encoders_cb(self, msg):
        now_ns = self.now_ns()
        self.encoders_rows.append({'time_ns': now_ns, 'time_s': now_ns / 1e9, 'values': list(msg.data)})

    def joint_state_cb(self, msg):
        now_ns = self.now_ns()
        self.joint_state_rows.append({
            'time_ns': now_ns,
            'time_s': now_ns / 1e9,
            'names': list(msg.name),
            'positions': list(msg.position),
        })

    def health_cb(self):
        official_count = sum(1 for row in self.command_rows if row['role'] == 'official_remote')
        executed_count = sum(1 for row in self.command_rows if row['role'] == 'executed_command')
        self.get_logger().info(
            f'samples: official={official_count}, executed={executed_count}, '
            f'status={len(self.status_rows)}, joints={len(self.joints_rows)}, '
            f'encoders={len(self.encoders_rows)}, joint_states={len(self.joint_state_rows)}'
        )

In [3]:
if not rclpy.ok():
    rclpy.init()

logger = LiveCommandAccuracyLogger()
deadline = time.monotonic() + ACQUISITION_DURATION_S
print(f'Starting acquisition for {ACQUISITION_DURATION_S:.1f} s. Start or keep rosbag play running now.')

try:
    while time.monotonic() < deadline:
        rclpy.spin_once(logger, timeout_sec=0.1)
except KeyboardInterrupt:
    print('Acquisition interrupted by user.')
finally:
    command_df = pd.DataFrame(logger.command_rows)
    status_df = pd.DataFrame(logger.status_rows)
    joints_df = pd.DataFrame(logger.joints_rows)
    encoders_df = pd.DataFrame(logger.encoders_rows)
    joint_states_df = pd.DataFrame(logger.joint_state_rows)
    logger.destroy_node()

command_df.to_csv(RAW_COMMAND_CSV, index=False)
print(f'Raw command samples written to: {RAW_COMMAND_CSV}')
display(command_df.head())

Starting acquisition for 170.0 s. Start or keep rosbag play running now.


[INFO] [1783434248.410296593] [te02_004_live_command_accuracy_logger]: samples: official=0, executed=0, status=472, joints=28, encoders=472, joint_states=28
[INFO] [1783434253.385983333] [te02_004_live_command_accuracy_logger]: samples: official=0, executed=0, status=1330, joints=78, encoders=1330, joint_states=78
[INFO] [1783434258.386378709] [te02_004_live_command_accuracy_logger]: samples: official=0, executed=0, status=2138, joints=128, encoders=2138, joint_states=128
[INFO] [1783434263.386365153] [te02_004_live_command_accuracy_logger]: samples: official=0, executed=0, status=2888, joints=178, encoders=2888, joint_states=178
[INFO] [1783434268.386576958] [te02_004_live_command_accuracy_logger]: samples: official=0, executed=0, status=3732, joints=228, encoders=3732, joint_states=228
[INFO] [1783434273.385907541] [te02_004_live_command_accuracy_logger]: samples: official=0, executed=0, status=4631, joints=277, encoders=4630, joint_states=277
[INFO] [1783434278.386968181] [te02_004_

Raw command samples written to: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004/te02_004_live_raw_commands.csv


""


In [7]:
def command_key(values):
    return tuple(int(v) for v in values)

def command_subset(df, role):
    if df.empty:
        return pd.DataFrame(columns=['time_ns', 'time_s', 'command', 'valid_shape'])
    return df.loc[df['role'] == role, ['time_ns', 'time_s', 'command', 'valid_shape']].sort_values('time_ns')

def match_commands(reference_df, executed_df):
    if reference_df.empty or executed_df.empty:
        return pd.DataFrame(columns=[
            'reference_time_s', 'executed_time_s', 'dt_ms', 'reference_command',
            'executed_command', 'exact_match', 'valid_shape'
        ])

    ref = reference_df.copy()
    exe = executed_df.copy()
    ref['reference_time_ns'] = ref['time_ns']
    exe['executed_time_ns'] = exe['time_ns']

    matched = pd.merge_asof(
        ref,
        exe,
        on='time_ns',
        direction='nearest',
        tolerance=int(COMMAND_MATCH_WINDOW_MS * 1e6),
        suffixes=('_reference', '_executed'),
    )

    rows = []
    for _, row in matched.iterrows():
        reference_command = row.get('command_reference')
        executed_command = row.get('command_executed')
        has_match = isinstance(executed_command, list)
        dt_ms = np.nan
        if has_match and not pd.isna(row.get('executed_time_ns')):
            dt_ms = (int(row['executed_time_ns']) - int(row['reference_time_ns'])) / 1e6
        exact_match = bool(has_match and command_key(reference_command) == command_key(executed_command))
        valid_shape = bool(row.get('valid_shape_reference', False) and row.get('valid_shape_executed', False))
        rows.append({
            'reference_time_s': row.get('time_s_reference'),
            'executed_time_s': row.get('time_s_executed'),
            'dt_ms': dt_ms,
            'reference_command': reference_command,
            'executed_command': executed_command if has_match else None,
            'exact_match': exact_match,
            'valid_shape': valid_shape,
        })
    return pd.DataFrame(rows)

official_df = command_subset(command_df, 'official_remote')
executed_df = command_subset(command_df, 'executed_command')
matched_df = match_commands(official_df, executed_df)
matched_df.to_csv(MATCHED_COMMAND_CSV, index=False)

display(matched_df.head(20))
print(f'Official command samples: {len(official_df)}')
print(f'Executed command samples: {len(executed_df)}')
print(f'Matched command samples: {len(matched_df)}')

,reference_time_s,executed_time_s,dt_ms,reference_command,executed_command,exact_match,valid_shape


Official command samples: 0
Executed command samples: 0
Matched command samples: 0


In [8]:
def signal_summary(topic, df):
    row = {'topic': topic, 'samples': int(len(df)), 'available': bool(len(df) > 0)}
    if len(df) > 1:
        duration = (df['time_ns'].max() - df['time_ns'].min()) / 1e9
        row['duration_s'] = duration
        row['rate_hz'] = (len(df) - 1) / duration if duration > 0 else np.nan
    else:
        row['duration_s'] = np.nan
        row['rate_hz'] = np.nan
    return row

supporting_df = pd.DataFrame([
    signal_summary(STATUS_TOPIC, status_df),
    signal_summary(JOINTS_TOPIC, joints_df),
    signal_summary(ENCODERS_TOPIC, encoders_df),
    signal_summary(JOINT_STATES_TOPIC, joint_states_df),
])

if not status_df.empty:
    supporting_df.loc[supporting_df['topic'] == STATUS_TOPIC, 'unique_values'] = str(sorted(status_df['value'].unique().tolist()))

supporting_df.to_csv(SUPPORTING_SIGNALS_CSV, index=False)
display(supporting_df)

,topic,samples,available,duration_s,rate_hz,unique_values
0,/statusjoy,27080,True,161.721279,167.442406,[1]
1,/joints,1616,True,161.625502,9.992235,NaN
2,/encoders,27079,True,161.720776,167.436743,NaN
3,/joint_states,1616,True,161.626290,9.992186,NaN


In [9]:
matched_count = int(len(matched_df))
exact_matches = int(matched_df['exact_match'].sum()) if matched_count else 0
valid_shape_count = int(matched_df['valid_shape'].sum()) if matched_count else 0
accuracy_pct = 100.0 * exact_matches / matched_count if matched_count else np.nan

if len(official_df) == 0 or len(executed_df) == 0:
    command_status = 'INCONCLUSIVE'
    command_reason = (
        f'Cannot validate 100% command accuracy because {OFFICIAL_REMOTE_TOPIC} has '
        f'{len(official_df)} received samples and {EXECUTED_COMMAND_TOPIC} has {len(executed_df)} received samples.'
    )
elif matched_count == 0:
    command_status = 'FAIL'
    command_reason = f'No command pairs matched within {COMMAND_MATCH_WINDOW_MS:.1f} ms.'
elif exact_matches == matched_count and valid_shape_count == matched_count:
    command_status = 'PASS'
    command_reason = 'Every matched command vector is valid and exactly equal to the official remote-controller vector.'
else:
    command_status = 'FAIL'
    command_reason = (
        f'{exact_matches}/{matched_count} command vectors are exactly equal; '
        f'{valid_shape_count}/{matched_count} command pairs have the expected vector shape.'
    )

summary_rows = [{
    'metric': 'command_vector_accuracy',
    'official_topic': OFFICIAL_REMOTE_TOPIC,
    'executed_topic': EXECUTED_COMMAND_TOPIC,
    'samples': matched_count,
    'accuracy_pct': accuracy_pct,
    'status': command_status,
    'reason': command_reason,
}]

for _, row in supporting_df.iterrows():
    summary_rows.append({
        'metric': f'supporting_signal:{row["topic"]}',
        'official_topic': '',
        'executed_topic': '',
        'samples': int(row['samples']),
        'accuracy_pct': np.nan,
        'status': 'AVAILABLE' if row['available'] else 'MISSING',
        'reason': f'rate_hz={row.get("rate_hz")}',
    })

summary_rows.append({
    'metric': 'TE02_004_overall',
    'official_topic': OFFICIAL_REMOTE_TOPIC,
    'executed_topic': EXECUTED_COMMAND_TOPIC,
    'samples': matched_count,
    'accuracy_pct': accuracy_pct,
    'status': command_status,
    'reason': command_reason,
})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f'Wrote summary: {SUMMARY_CSV}')
print(f'Overall TE02_004 status: {command_status}')

,metric,official_topic,executed_topic,samples,accuracy_pct,status,reason
0,command_vector_accuracy,/telejoy,/telejoyout,0,NaN,INCONCLUSIVE,Cannot validate 100% command accuracy because ...
1,supporting_signal:/statusjoy,,,27080,NaN,AVAILABLE,rate_hz=167.44240601102175
2,supporting_signal:/joints,,,1616,NaN,AVAILABLE,rate_hz=9.992235029209088
3,supporting_signal:/encoders,,,27079,NaN,AVAILABLE,rate_hz=167.43674285092013
4,supporting_signal:/joint_states,,,1616,NaN,AVAILABLE,rate_hz=9.992186265945904
5,TE02_004_overall,/telejoy,/telejoyout,0,NaN,INCONCLUSIVE,Cannot validate 100% command accuracy because ...


Wrote summary: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004/te02_004_live_command_accuracy_summary.csv
Overall TE02_004 status: INCONCLUSIVE
